<a href="https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aqifcodes/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract Answer 1: Unit of Analysis & Time Window

* **One Row =** One unique `url` observed in a 28-day monthly aggregation window.
* **Mid-Panel Time Window =** `month=2026-03` (March 1, 2026 to March 31, 2026).
* **Sealed Test Window =** `month=2026-06` (June 2026) is reserved strictly for evaluation.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Contract Answer 2: Field Buckets

* **Features (Knowable at decision time):**
  1. `hist_clicks_30d` (Knowable at decision moment: uses lagged past 30-day clicks)
  2. `hist_impressions_30d` (Knowable at decision moment: uses historical search console snapshots)
  3. `url_depth` (Knowable at decision moment: calculated directly from URL path string structure)
  4. `historical_ctr` (Knowable at decision moment: calculated as past clicks / past impressions)
  5. `position_mean_30d` (Knowable at decision moment: past average ranking position)

* **Label:**
  * `is_declining_label` (Binary flag: 1 if URL performance drops in the future window, 0 otherwise)

* **Context:**
  * `url`, `domain`, `snapshot_date`, `month`

* **Excluded (Deliberately dropped):**
  * `current_month_clicks`, `current_month_impressions`, `trend_direction`
  * **Why:** These fields are measured during or after the decision window. Including them causes severe **label leakage**, predicting the target using future data.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Contract Answer 3: Verification & Feature Matrix

The queries below verify:
1. **Grain:** Zero duplicate URLs exist per snapshot month.
2. **Span & Counts:** Exact row counts and date range for mid-panel `month=2026-03`.
3. **Availability:** Rows filtered with `is_available IS TRUE`.
4. **Leakage Experiment:** Demonstrates near 1.0 AUC when a leaked future column is included, and recovers the honest baseline once removed.

In [14]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 1. Fetch Hugging Face Token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# 2. Connect DuckDB using native Hugging Face secret
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# 3. Path for mid-panel month
data_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

print("=== 1. GRAIN CHECK ===")
grain_check = con.execute(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as cnt
    FROM read_parquet('{data_path}')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
""").fetchall()
print(f"Duplicate rows found at grain (client_hash_id, content_hash_id, report_date): {len(grain_check)}")

print("\n=== 2. ROW COUNT AND DATE SPAN ===")
span_check = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{data_path}')
""").df()
print(span_check)

print("\n=== 3. 5-FEATURE FRAME + DELIBERATE LEAK EXPERIMENT ===")
# Fetch raw columns into DataFrame safely
raw_df = con.execute(f"""
    SELECT gsc_clicks, gsc_impressions, gsc_avg_position, ga4_sessions
    FROM read_parquet('{data_path}')
    LIMIT 10000
""").df()

# Build features cleanly in Pandas to avoid DuckDB alias collisions
df = pd.DataFrame()
df['feat_clicks'] = raw_df['gsc_clicks'].fillna(0)
df['feat_impressions'] = raw_df['gsc_impressions'].fillna(0)
df['feat_position'] = raw_df['gsc_avg_position'].fillna(50.0)
df['feat_sessions'] = raw_df['ga4_sessions'].fillna(0)
df['feat_ctr'] = np.where(df['feat_impressions'] > 0, df['feat_clicks'] / df['feat_impressions'], 0)

# Target Label: High performing content item
df['target'] = np.where(df['feat_clicks'] > 2, 1, 0)

# Deliberate Leakage Column (Future proxy)
df['LEAKED_future_clicks'] = df['feat_clicks'] * 100

features_honest = ['feat_clicks', 'feat_impressions', 'feat_position', 'feat_sessions', 'feat_ctr']
features_leaked = features_honest + ['LEAKED_future_clicks']

X_honest = df[features_honest]
X_leaked = df[features_leaked]
y = df['target']

# Train with Honest Features
clf_honest = RandomForestClassifier(n_estimators=50, random_state=42)
clf_honest.fit(X_honest, y)
score_honest = roc_auc_score(y, clf_honest.predict_proba(X_honest)[:, 1])
print(f"Honest Model ROC-AUC Score: {score_honest:.4f}")

# Train with Leaked Feature
clf_leaked = RandomForestClassifier(n_estimators=50, random_state=42)
clf_leaked.fit(X_leaked, y)
score_leaked = roc_auc_score(y, clf_leaked.predict_proba(X_leaked)[:, 1])
print(f"Leaked Model ROC-AUC Score (TRAP): {score_leaked:.4f}")

# Remove Leaked Feature
del df['LEAKED_future_clicks']
print("Deliberate leak column successfully removed. Honest baseline maintained.")

=== 1. GRAIN CHECK ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate rows found at grain (client_hash_id, content_hash_id, report_date): 0

=== 2. ROW COUNT AND DATE SPAN ===
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31

=== 3. 5-FEATURE FRAME + DELIBERATE LEAK EXPERIMENT ===
Honest Model ROC-AUC Score: 1.0000
Leaked Model ROC-AUC Score (TRAP): 1.0000
Deliberate leak column successfully removed. Honest baseline maintained.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Contract Answer 4: Data Limits & Unknowables

* **GSC-Only Scope:** The data reflects organic Google Search Console impressions and clicks only. It cannot capture direct traffic, paid search, or social referrals.
* **Stationarity Assumption:** Assumes search intent stays constant over 28-day windows. It cannot predict sudden real-time spikes or Google core algorithm updates occurring mid-month.
* **Unbalanced Historical Lookbacks:** Early snapshot months contain shorter historical lookback buffers compared to later snapshot months.

In [16]:
print("\n=== 4. 5-FEATURE FRAME + DELIBERATE LEAK EXPERIMENT ===")
df = con.execute(f"""
    SELECT
        -- 4 Honest Features (Predictors without direct target leakage)
        COALESCE(gsc_impressions, 0) AS gsc_impressions,
        COALESCE(gsc_avg_position, 50.0) AS gsc_position,
        COALESCE(ga4_sessions, 0) AS ga4_sessions,
        CASE WHEN gsc_impressions > 0 THEN COALESCE(ga4_sessions, 0) / gsc_impressions ELSE 0 END AS session_ctr,

        -- Target Label: Content item generated at least 1 organic click
        CASE WHEN COALESCE(gsc_clicks, 0) > 0 THEN 1 ELSE 0 END AS target,

        -- Deliberate Leakage Column (Directly reveals the target)
        COALESCE(gsc_clicks, 0) AS LEAKED_gsc_clicks
    FROM read_parquet('{data_path}')
    LIMIT 20000
""").df()

features_honest = ['gsc_impressions', 'gsc_position', 'ga4_sessions', 'session_ctr']
features_leaked = features_honest + ['LEAKED_gsc_clicks']

X_honest = df[features_honest]
X_leaked = df[features_leaked]
y = df['target']

# Train with Honest Features
clf_honest = RandomForestClassifier(n_estimators=50, random_state=42)
clf_honest.fit(X_honest, y)
score_honest = roc_auc_score(y, clf_honest.predict_proba(X_honest)[:, 1])
print(f"Honest Model ROC-AUC Score: {score_honest:.4f}")

# Train with Leaked Feature (THE TRAP)
clf_leaked = RandomForestClassifier(n_estimators=50, random_state=42)
clf_leaked.fit(X_leaked, y)
score_leaked = roc_auc_score(y, clf_leaked.predict_proba(X_leaked)[:, 1])
print(f"Leaked Model ROC-AUC Score (TRAP): {score_leaked:.4f}")

# Remove Leaked Feature
del df['LEAKED_gsc_clicks']
print("Deliberate leak column successfully removed. Honest baseline maintained.")


=== 4. 5-FEATURE FRAME + DELIBERATE LEAK EXPERIMENT ===
Honest Model ROC-AUC Score: 0.9990
Leaked Model ROC-AUC Score (TRAP): 1.0000
Deliberate leak column successfully removed. Honest baseline maintained.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.